# 07 — Evaluation der drei LLM-Pipelines

Vergleicht die Erklärungsqualität von:
- **Pipeline 04** (JSON → Text): strukturierte Daten als Input
- **Pipeline 05** (Vision → Text): Waterfall-Plot als Input
- **Pipeline 06** (Tool-Use): LLM fragt Daten aktiv ab

**Drei Bewertungsebenen:**
1. Quantitativ (kein API-Key nötig): Länge, Token-Verbrauch, Kosten, Laufzeit
2. Faithfulness-Check (kein API-Key): Erwähnt die Erklärung die tatsächlich wichtigsten Features?
3. LLM-as-Judge (API-Key nötig): Strukturierte Bewertung nach Faithfulness, Verständlichkeit, Vollständigkeit

In [1]:
from __future__ import annotations

import sys, json, re, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display

from utils import INSTANCE_IDS, RESULTS_DIR, EXPLANATIONS_DIR, PROMPTS_DIR
from utils.llm import ask_text, DEFAULT_MODEL

LOSS_KEY = 'poisson_log'
MODEL    = DEFAULT_MODEL
PIPELINES = ['04', '05', '06']
PIPELINE_LABELS = {'04': 'JSON→Text', '05': 'Vision', '06': 'Tool-Use'}
XAI_MODELS = ['xgb', 'ebm']

# Kosten pro 1M Token (claude-sonnet-4-6, Stand Mai 2025)
COST_INPUT_PER_M       = 3.00   # USD – reguläre Input-Tokens
COST_CACHE_READ_PER_M  = 0.30   # USD – Cache-Read-Tokens (90 % Rabatt)
COST_OUTPUT_PER_M      = 15.00  # USD – Output-Tokens

print('Setup abgeschlossen.')

Setup abgeschlossen.


## 1 · Ergebnisse laden

In [2]:
records = []
for pipeline in PIPELINES:
    p = RESULTS_DIR / f'pipeline{pipeline}'
    for xai in XAI_MODELS:
        for iid in INSTANCE_IDS:
            f = p / f'{xai}_inst{iid}.json'
            if not f.exists():
                print(f'FEHLT: {f}')
                continue
            d = json.loads(f.read_text())
            usage = d.get('usage', {})
            in_tok   = usage.get('input_tokens', 0)
            out_tok  = usage.get('output_tokens', 0)
            cache_r  = usage.get('cache_read_input_tokens', 0)
            # Cache-Read-Tokens kosten nur $0.30/M statt $3.00/M
            regular_in = max(in_tok - cache_r, 0)
            cost = (
                regular_in * COST_INPUT_PER_M
                + cache_r  * COST_CACHE_READ_PER_M
                + out_tok  * COST_OUTPUT_PER_M
            ) / 1_000_000
            records.append({
                'pipeline':      pipeline,
                'pipeline_label': PIPELINE_LABELS[pipeline],
                'xai_model':     xai.upper(),
                'instance_id':   iid,
                'explanation':   d.get('explanation', ''),
                'word_count':    len(d.get('explanation', '').split()),
                'tok_input':     in_tok,
                'tok_output':    out_tok,
                'tok_cache':     cache_r,
                'tok_total':     in_tok + out_tok,
                'cost_usd':      round(cost, 5),
                'elapsed_s':     d.get('elapsed_s', 0),
                'n_tool_calls':  d.get('n_tool_calls', 0),
                'y_true':        d.get('y_true', None),
                'prediction':    d.get('prediction', None),
            })

df = pd.DataFrame(records)
print(f'{len(df)} Ergebnisse geladen.')
display(df.groupby(['pipeline_label', 'xai_model']).size().unstack())

60 Ergebnisse geladen.


xai_model,EBM,XGB
pipeline_label,,
JSON→Text,10,10
Tool-Use,10,10
Vision,10,10


In [3]:
# Instanzen-Stichprobe dokumentieren
# Stratifizierte Auswahl über cnt-Quintile (keine Extremfälle: cnt>=31, kein Gewitter, kein Feiertagscluster)
instance_doc = [
    {"instance_id": 224,  "cnt_true":  51, "note": "Do 02M 13h, klar, ~8°C, 2011"},
    {"instance_id": 580,  "cnt_true": 159, "note": "So 03M 17h, klar, ~13°C, 2011"},
    {"instance_id": 1041, "cnt_true": 251, "note": "So 05M 10h, bewölkt, ~27°C, 2011"},
    {"instance_id": 1481, "cnt_true": 114, "note": "Sa 07M 08h, klar, ~32°C, 2011"},
    {"instance_id": 1677, "cnt_true": 410, "note": "Fr 08M 18h, bewölkt, ~30°C, 2011"},
    {"instance_id": 2058, "cnt_true":  31, "note": "Fr 10M 05h, klar, ~14°C, 2011"},
    {"instance_id": 2510, "cnt_true":  89, "note": "Mi 12M 10h, bewölkt, ~20°C, 2011"},
    {"instance_id": 3543, "cnt_true": 191, "note": "So 05M 09h, bewölkt, ~21°C, 2012"},
    {"instance_id": 3847, "cnt_true": 302, "note": "So 06M 20h, klar, ~25°C, 2012"},
    {"instance_id": 4454, "cnt_true": 557, "note": "Mi 09M 07h, klar, ~21°C, 2012"},
]
inst_df = pd.DataFrame(instance_doc)
print("Ausgewählte Test-Instanzen (stratifiziertes Quantil-Sampling):")
print("  cnt-Bereich: 31–557, verteilt über 5 cnt-Quintile")
print("  Kein cnt<=20, kein Gewitter (weathersit=4), beide Jahre (2011/2012)")
print()
display(inst_df)


Ausgewählte Test-Instanzen (stratifiziertes Quantil-Sampling):
  cnt-Bereich: 31–557, verteilt über 5 cnt-Quintile
  Kein cnt<=20, kein Gewitter (weathersit=4), beide Jahre (2011/2012)



,instance_id,cnt_true,note
0,224,51,"Do 02M 13h, klar, ~8°C, 2011"
1,580,159,"So 03M 17h, klar, ~13°C, 2011"
2,1041,251,"So 05M 10h, bewölkt, ~27°C, 2011"
3,1481,114,"Sa 07M 08h, klar, ~32°C, 2011"
4,1677,410,"Fr 08M 18h, bewölkt, ~30°C, 2011"
5,2058,31,"Fr 10M 05h, klar, ~14°C, 2011"
6,2510,89,"Mi 12M 10h, bewölkt, ~20°C, 2011"
7,3543,191,"So 05M 09h, bewölkt, ~21°C, 2012"
8,3847,302,"So 06M 20h, klar, ~25°C, 2012"
9,4454,557,"Mi 09M 07h, klar, ~21°C, 2012"


## 2 · Quantitative Analyse

In [4]:
quant = df.groupby('pipeline_label').agg(
    Wörter_mean  =('word_count',  'mean'),
    Wörter_std   =('word_count',  'std'),
    tok_input    =('tok_input',   'mean'),
    tok_output   =('tok_output',  'mean'),
    tok_total    =('tok_total',   'mean'),
    Kosten_USD   =('cost_usd',    'sum'),
    Zeit_s       =('elapsed_s',   'mean'),
    Tool_Calls   =('n_tool_calls','mean'),
).round(2)
print('Quantitativer Vergleich (Mittelwert pro Pipeline):')
display(quant)

Quantitativer Vergleich (Mittelwert pro Pipeline):


,Wörter_mean,Wörter_std,tok_input,tok_output,tok_total,Kosten_USD,Zeit_s,Tool_Calls
pipeline_label,,,,,,,,
JSON→Text,207.85,10.32,616.15,510.15,1126.30,0.16,11.73,0.00
Tool-Use,305.05,20.50,3489.25,1224.50,4713.75,0.58,28.76,5.65
Vision,211.60,10.17,2166.50,528.10,2694.60,0.29,12.32,0.00


In [5]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
labels = [PIPELINE_LABELS[p] for p in PIPELINES]
colors = ['#4c72b0', '#dd8452', '#55a868']

# Wortanzahl
vals = [df[df.pipeline == p]['word_count'].mean() for p in PIPELINES]
axes[0].bar(labels, vals, color=colors)
axes[0].set_title('Ø Wortanzahl')
axes[0].set_ylabel('Wörter')

# Token-Verbrauch
in_vals  = [df[df.pipeline == p]['tok_input'].mean()  for p in PIPELINES]
out_vals = [df[df.pipeline == p]['tok_output'].mean() for p in PIPELINES]
x = range(len(labels))
axes[1].bar(x, in_vals,  label='Input',  color=colors, alpha=0.5)
axes[1].bar(x, out_vals, label='Output', color=colors, bottom=in_vals)
axes[1].set_xticks(list(x)); axes[1].set_xticklabels(labels)
axes[1].set_title('Ø Token-Verbrauch')
axes[1].set_ylabel('Tokens')
axes[1].legend()

# Kosten gesamt
vals = [df[df.pipeline == p]['cost_usd'].sum() for p in PIPELINES]
axes[2].bar(labels, vals, color=colors)
axes[2].set_title('Gesamtkosten (10 Calls)')
axes[2].set_ylabel('USD')

plt.suptitle('Quantitativer Vergleich der drei LLM-Pipelines', y=1.02)
plt.tight_layout()
out_path = RESULTS_DIR / 'eval_quantitative.png'
plt.savefig(out_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Gespeichert: {out_path}')

Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_XAI_Stahl_SoSe26/results/eval_quantitative.png


/var/folders/z8/rlz8vq9j2cs6wj25qj_g6y700000gn/T/ipykernel_69734/4182671026.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3 · Faithfulness-Check (keyword-basiert)

In [ ]:
def top3_with_contributions(xai_model: str, instance_id: int, top_k: int = 3) -> list[dict]:
    """Gibt Top-k-Features mit Beitrag und Richtung aus der lokalen Erklärungsdatei zurück."""
    path = EXPLANATIONS_DIR / f"local_{xai_model.lower()}_{LOSS_KEY}_inst{instance_id}.json"
    d = json.loads(path.read_text())
    return [
        {"feature": c["feature"], "contribution": c["contribution"]}
        for c in d["contributions"][:top_k]
    ]


def llm_faithfulness_check(explanation: str, top3: list[dict]) -> tuple[float, list[dict]]:
    """
    Nutzt Claude, um zu prüfen ob die Top-3-Features korrekt mit richtiger
    Wirkungsrichtung im Erklärungstext erwähnt werden.
    Score: 1.0 pro Feature korrekt (erwähnt + Richtung), 0.5 erwähnt ohne korrekte Richtung.
    """
    features_desc = "\n".join(
        f"  - {e['feature']}: Beitrag={e['contribution']:+.4f} "
        f"({'erhöht' if e['contribution'] > 0 else 'senkt'} Vorhersage)"
        for e in top3
    )
    prompt = (
        "Prüfe, ob die folgende Modellerklärung die drei wichtigsten Features "
        "korrekt mit richtiger Wirkungsrichtung wiedergibt.\n\n"
        f"Top-3-Features laut Modell:\n{features_desc}\n\n"
        "Erklärungstext:\n"
        f'"""\n{explanation}\n"""\n\n'
        "Für jedes Feature: Wird es (oder ein klares Synonym) erwähnt? "
        "Ist die Wirkungsrichtung korrekt dargestellt?\n\n"
        "Antworte ausschließlich in diesem Format (eine Zeile pro Feature, kein weiterer Text):\n"
        "FEATURE_1: <feature_name> | ERWAEHNT: ja/nein | RICHTUNG: korrekt/falsch/na\n"
        "FEATURE_2: <feature_name> | ERWAEHNT: ja/nein | RICHTUNG: korrekt/falsch/na\n"
        "FEATURE_3: <feature_name> | ERWAEHNT: ja/nein | RICHTUNG: korrekt/falsch/na"
    )

    assessments = []
    try:
        resp = ask_text(prompt, max_tokens=300)
        raw = resp["content"][0]["text"]
        for line in raw.strip().splitlines():
            m_e = re.search(r'ERWAEHNT:\s*(ja|nein)', line, re.IGNORECASE)
            m_r = re.search(r'RICHTUNG:\s*(korrekt|falsch|na)', line, re.IGNORECASE)
            m_f = re.search(r'FEATURE_\d:\s*([^|]+)', line)
            if m_e:
                mentioned = m_e.group(1).lower() == 'ja'
                val = m_r.group(1).lower() if m_r else 'na'
                direction_correct = True if val == 'korrekt' else (False if val == 'falsch' else None)
                assessments.append({
                    "feature": m_f.group(1).strip() if m_f else "",
                    "mentioned": mentioned,
                    "direction_correct": direction_correct,
                })
    except Exception:
        return 0.0, []

    score = sum(
        1.0 if a.get("mentioned") and a.get("direction_correct") else
        0.5 if a.get("mentioned") else
        0.0
        for a in assessments
    )
    return round(score / max(len(top3), 1), 3), assessments


_faith_cache = RESULTS_DIR / 'eval_faithfulness_check.json'

if _faith_cache.exists():
    faith_rows = json.loads(_faith_cache.read_text())
    print(f'Faithfulness-Check aus Cache geladen: {len(faith_rows)} Einträge')
    print('(Cache löschen zum Neuberechnen)')
else:
    print("Starte LLM-Faithfulness-Check …")
    faith_rows = []
    for _, row in df.iterrows():
        top3 = top3_with_contributions(row["xai_model"], row["instance_id"])
        score, assessments = llm_faithfulness_check(row["explanation"], top3)
        faith_rows.append({
            "pipeline_label":  row["pipeline_label"],
            "xai_model":       row["xai_model"],
            "instance_id":     row["instance_id"],
            "top3_features":   [e["feature"] for e in top3],
            "faithfulness":    score,
            "llm_assessments": assessments,
        })
    _faith_cache.write_text(json.dumps(faith_rows, indent=2, ensure_ascii=False))
    print(f'Gespeichert: {_faith_cache.name}')

faith_df = pd.DataFrame(faith_rows)

faith_summary = faith_df.groupby("pipeline_label")["faithfulness"].agg(["mean", "std"]).round(3)
faith_summary.columns = ["Ø Faithfulness", "Std"]
print("Faithfulness (0–1, LLM-basiert): korrekt erwähnte Top-3-Features mit richtiger Richtung:")
display(faith_summary)

print()
print("Details:")
display(faith_df[["pipeline_label","xai_model","instance_id","top3_features","faithfulness"]])

## 4 · LLM-as-Judge Evaluation

> **Voraussetzung:** `ANTHROPIC_API_KEY` in `.env`.

Claude bewertet jede Erklärung auf drei Dimensionen (1–5):
- **Faithfulness**: Werden die wichtigsten Treiber korrekt beschrieben?
- **Clarity**: Ist die Erklärung für Nicht-Experten verständlich?
- **Completeness**: Werden Vorhersage, Treiber und praktische Implikation abgedeckt?

In [7]:
JUDGE_SYSTEM = (PROMPTS_DIR / "judge_system.md").read_text()


WEEKDAYS_JUDGE = {0:"Sonntag", 1:"Montag", 2:"Dienstag", 3:"Mittwoch",
                  4:"Donnerstag", 5:"Freitag", 6:"Samstag"}
MONTHS_JUDGE   = {1:"Januar", 2:"Februar", 3:"März", 4:"April", 5:"Mai",
                  6:"Juni", 7:"Juli", 8:"August", 9:"September",
                  10:"Oktober", 11:"November", 12:"Dezember"}
WEATHER_JUDGE  = {1:"klar/wenige Wolken", 2:"Nebel/bewölkt",
                  3:"leichter Regen/Schnee", 4:"Starkregen/Gewitter"}


def build_judge_prompt(row: dict, xai_model: str, instance_id: int) -> str:
    local_path = EXPLANATIONS_DIR / f"local_{xai_model.lower()}_{LOSS_KEY}_inst{instance_id}.json"
    l = json.loads(local_path.read_text())
    fv = l["feature_values"]

    top3 = [{"feature": c["feature"], "contribution": c["contribution"],
              "value": c["value"]}
             for c in l["contributions"][:3]]

    # Menschenlesbare Feature-Werte für den Judge
    fv_readable = {
        "uhrzeit":           f"{int(fv['hr']):02d}:00 Uhr",
        "wochentag":         WEEKDAYS_JUDGE.get(int(fv["weekday"]), str(fv["weekday"])),
        "monat":             MONTHS_JUDGE.get(int(fv["mnth"]), str(fv["mnth"])),
        "jahr":              "2011" if int(fv["yr"]) == 0 else "2012",
        "wetter":            WEATHER_JUDGE.get(int(fv["weathersit"]), str(fv["weathersit"])),
        "temperatur_celsius": f"~{float(fv['temp']) * 41:.1f} °C",
        "luftfeuchtigkeit":  f"{float(fv['hum']) * 100:.0f} %",
        "windgeschwindigkeit": f"{float(fv['windspeed']) * 67:.1f} km/h",
        "feiertag":          "ja" if int(fv["holiday"]) == 1 else "nein",
    }

    ground_truth = {
        "model":                  xai_model,
        "prediction":             l["prediction"],
        "y_true":                 l["y_true"],
        "top3_drivers":           top3,
        "feature_values_readable": fv_readable,
    }

    # Für Tool-Use: echten Tool-Call-Trace einbetten
    pipeline = row.get("pipeline", "")
    if "06" in pipeline or pipeline == "06_tooluse":
        trace_path = RESULTS_DIR / f"pipeline06/{xai_model.lower()}_inst{instance_id}.json"
        if trace_path.exists():
            trace_data = json.loads(trace_path.read_text())
            ground_truth["tool_call_trace"] = [
                {
                    "round":     i + 1,
                    "tool":      c["tool"],
                    "arguments": c["arguments"],
                    "result":    c["result_preview"],
                }
                for i, c in enumerate(trace_data.get("tool_calls", []))
            ]
            ground_truth["tool_trace_note"] = (
                "Die abgerufenen Werte (Beiträge, Percentile, Counterfactuals) "
                "sind korrekt und dürfen als Belege für Faithfulness gewertet werden."
            )

    output_instruction = (
        "Antworte ausschließlich in diesem Format (kein JSON, kein Markdown, kein sonstiger Text):\n"
        "FAITHFULNESS: <1-5>\n"
        "CLARITY: <1-5>\n"
        "COMPLETENESS: <1-5>\n"
        "FAITHFULNESS_REASONING: <1 Satz>\n"
        "CLARITY_REASONING: <1 Satz>\n"
        "COMPLETENESS_REASONING: <1 Satz>"
    )
    return json.dumps({
        "task": (
            "Bewerte die folgende Erklärung nach der definierten Rubrik. "
            "Vergib für jedes Kriterium einen Score (1–5) und begründe kurz."
        ),
        "ground_truth": ground_truth,
        "explanation": row["explanation"],
        "output_format": output_instruction,
    }, ensure_ascii=False, indent=2)


In [8]:
def _parse_judge_response(raw: str) -> dict:
    """Extrahiert Judge-Scores robust aus Plaintext, JSON oder Markdown-Codeblock."""
    # Markdown-Codeblock entfernen (Modell ignoriert manchmal 'kein Markdown')
    code_block = re.search(r'```(?:json)?\s*(.*?)(?:```|$)', raw, re.DOTALL)
    inner = code_block.group(1).strip() if code_block else raw

    scores = {}

    # Vollständiges JSON versuchen
    try:
        json_match = re.search(r'\{.*\}', inner, re.DOTALL)
        if json_match:
            d = json.loads(json_match.group())
            d_up = {k.upper(): v for k, v in d.items()}
            for key in ['FAITHFULNESS', 'CLARITY', 'COMPLETENESS']:
                if key in d_up:
                    scores[key.lower()] = int(d_up[key])
            for key in ['FAITHFULNESS_REASONING', 'CLARITY_REASONING', 'COMPLETENESS_REASONING']:
                base = key.replace('_REASONING', '').lower()
                if key in d_up:
                    scores[f'{base}_reasoning'] = str(d_up[key])
            if len(scores) >= 3:
                return scores
    except (json.JSONDecodeError, ValueError):
        pass

    # Fallback: Regex auf JSON-Keys (auch bei abgeschnittenem JSON)
    for key in ['FAITHFULNESS', 'CLARITY', 'COMPLETENESS']:
        m = re.search(rf'"?{key}"?\s*:\s*(\d)', inner, re.IGNORECASE)
        if m:
            scores[key.lower()] = int(m.group(1))
    for key in ['FAITHFULNESS_REASONING', 'CLARITY_REASONING', 'COMPLETENESS_REASONING']:
        base = key.replace('_REASONING', '').lower()
        m = re.search(rf'"?{key}"?\s*:\s*"([^"]+)', inner, re.IGNORECASE)
        if m:
            scores[f'{base}_reasoning'] = m.group(1)

    return scores


# Ergebnisse aus Cache laden falls vorhanden (v1-Judge-Ergebnisse)
_v1_cache = RESULTS_DIR / 'eval_llm_judge.json'
if _v1_cache.exists():
    judge_df = pd.DataFrame(json.loads(_v1_cache.read_text()))
    for col in ['faithfulness', 'clarity', 'completeness']:
        judge_df[col] = pd.to_numeric(judge_df[col], errors='coerce')
    print(f'v1-Ergebnisse aus Cache geladen: {len(judge_df)} Einträge')
    print('(Zelle überspringt API-Calls – entferne die Cache-Datei zum Neuberechnen)')
else:
    # Alle 60 Judge-Calls ausführen
    judge_rows = []
    total_in, total_out = 0, 0
    MAX_RETRIES = 3

    for _, row in df.iterrows():
        prompt = build_judge_prompt(
            row.to_dict(), row['xai_model'], row['instance_id']
        )

        scores = {}
        in_tok, out_tok = 0, 0
        raw = ''
        for attempt in range(MAX_RETRIES):
            response = ask_text(
                prompt,
                system=JUDGE_SYSTEM,
                model=MODEL,
                max_tokens=600,
                cache_system=True,
            )
            usage   = response.get('usage', {})
            in_tok  = usage.get('input_tokens', 0)
            out_tok = usage.get('output_tokens', 0)
            raw = response['content'][0]['text'].strip()
            scores = _parse_judge_response(raw)
            if all(scores.get(k) is not None for k in ['faithfulness', 'clarity', 'completeness']):
                break
            print(f"  ⚠️  Retry {attempt+1}/{MAX_RETRIES}: {row['pipeline_label']} {row['xai_model']} inst={row['instance_id']}")

        total_in += in_tok; total_out += out_tok

        judge_rows.append({
            'pipeline_label': row['pipeline_label'],
            'xai_model':      row['xai_model'],
            'instance_id':    row['instance_id'],
            'faithfulness':   scores.get('faithfulness', None),
            'clarity':        scores.get('clarity', None),
            'completeness':   scores.get('completeness', None),
            'reasoning': {
                'faithfulness':  scores.get('faithfulness_reasoning', ''),
                'clarity':       scores.get('clarity_reasoning', ''),
                'completeness':  scores.get('completeness_reasoning', ''),
            },
            'raw_response':   raw,
        })
        print(f"  {row['pipeline_label']:12s} {row['xai_model']} inst={row['instance_id']:4d}  "
              f"F={scores.get('faithfulness','?')} "
              f"C={scores.get('clarity','?')} "
              f"Co={scores.get('completeness','?')}  in={in_tok}")

    judge_df = pd.DataFrame(judge_rows)
    (RESULTS_DIR / 'eval_llm_judge.json').write_text(
        json.dumps(judge_rows, indent=2, ensure_ascii=False)
    )
    print(f'\nGesamt: input={total_in}  output={total_out}')

v1-Ergebnisse aus Cache geladen: 60 Einträge
(Zelle überspringt API-Calls – entferne die Cache-Datei zum Neuberechnen)


## 5 · Ergebnisse des LLM-Judges

In [9]:
# Fehlende Scores dokumentieren (JSON-Parse-Fehler beim Judge-Output)
null_mask = judge_df[['faithfulness', 'clarity', 'completeness']].isnull().any(axis=1)
if null_mask.any():
    print(f"⚠️  {null_mask.sum()} Einträge mit None-Scores (JSON-Parsing fehlgeschlagen):")
    display(judge_df[null_mask][['pipeline_label', 'xai_model', 'instance_id']])
    print("   → Diese Einträge werden von .mean() automatisch ausgeschlossen (pandas NaN-Handling).")
    print()

judge_summary = judge_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']].mean().round(2)
judge_counts  = judge_df.groupby('pipeline_label')[['faithfulness']].count().rename(
    columns={'faithfulness': 'n (gültig)'}
)
judge_summary = judge_summary.join(judge_counts)
print('Ø Judge-Scores (1–5) pro Pipeline  [None-Einträge aus Mittelwert ausgeschlossen]:')
display(judge_summary)

print()
judge_xai = judge_df.groupby(['pipeline_label', 'xai_model'])[['faithfulness', 'clarity', 'completeness']].mean().round(2)
print('Aufgeschlüsselt nach Modell:')
display(judge_xai)


Ø Judge-Scores (1–5) pro Pipeline  [None-Einträge aus Mittelwert ausgeschlossen]:


,faithfulness,clarity,completeness,n (gültig)
pipeline_label,,,,
JSON→Text,4.35,4.9,5.00,20
Tool-Use,4.50,3.9,4.95,20
Vision,3.80,4.6,4.90,20



Aufgeschlüsselt nach Modell:


faithfulness  clarity  completeness
pipeline_label xai_model                                     
JSON→Text      EBM                 4.4      4.9           5.0
               XGB                 4.3      4.9           5.0
Tool-Use       EBM                 4.7      3.9           5.0
               XGB                 4.3      3.9           4.9
Vision         EBM                 3.5      4.5           5.0
               XGB                 4.1      4.7           4.8

In [10]:
# Radar-Chart
from matplotlib.patches import FancyArrowPatch
import numpy as np

criteria   = ['Faithfulness', 'Clarity', 'Completeness']
n          = len(criteria)
angles     = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
angles    += angles[:1]
pipeline_colors = {'JSON→Text': '#4c72b0', 'Vision': '#dd8452', 'Tool-Use': '#55a868'}

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

for label, color in pipeline_colors.items():
    row = judge_summary.loc[label] if label in judge_summary.index else None
    if row is None: continue
    values = [row['faithfulness'], row['clarity'], row['completeness']]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=label, color=color)
    ax.fill(angles, values, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(criteria, fontsize=12)
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(['1','2','3','4','5'], fontsize=8)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.set_title('LLM-Judge Scores (1–5) pro Pipeline', pad=20)

plt.tight_layout()
out_path = RESULTS_DIR / 'eval_radar.png'
plt.savefig(out_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Gespeichert: {out_path}')

Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_XAI_Stahl_SoSe26/results/eval_radar.png


/var/folders/z8/rlz8vq9j2cs6wj25qj_g6y700000gn/T/ipykernel_69734/843577156.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6 · Gesamtübersicht und Empfehlung

In [11]:
# Kombinierte Tabelle: quantitativ + LLM-Judge
quant_p = df.groupby('pipeline_label').agg(
    Wörter    =('word_count', 'mean'),
    Tokens_in =('tok_input',  'mean'),
    Tokens_out=('tok_output', 'mean'),
    Kosten_USD=('cost_usd',   'sum'),
    Zeit_s    =('elapsed_s',  'mean'),
    ToolCalls =('n_tool_calls','mean'),
).round(2)

faith_p = faith_df.groupby('pipeline_label')['faithfulness'].mean().round(3)
faith_p.name = 'Faith_KW'

combined = quant_p.join(faith_p)
if len(judge_df) > 0:
    judge_p = judge_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']].mean().round(2)
    judge_p.columns = ['Judge_Faith', 'Judge_Clarity', 'Judge_Complete']
    judge_std = judge_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']].std().round(3)
    judge_std.columns = ['Judge_Faith_std', 'Judge_Clarity_std', 'Judge_Complete_std']
    judge_n = judge_df.groupby('pipeline_label')[['faithfulness']].count().rename(
        columns={'faithfulness': 'Judge_n'}
    )
    combined = combined.join(judge_p).join(judge_std).join(judge_n)

print('Gesamtübersicht:')
display(combined)

combined.to_csv(RESULTS_DIR / 'eval_summary.csv')
print(f'Gespeichert: {RESULTS_DIR / "eval_summary.csv"}')


Gesamtübersicht:


,Wörter,Tokens_in,Tokens_out,Kosten_USD,Zeit_s,ToolCalls,Faith_KW,Judge_Faith,Judge_Clarity,Judge_Complete,Judge_Faith_std,Judge_Clarity_std,Judge_Complete_std,Judge_n
pipeline_label,,,,,,,,,,,,,,
JSON→Text,207.85,616.15,510.15,0.16,11.73,0.00,0.942,4.35,4.9,5.00,0.745,0.308,0.000,20
Tool-Use,305.05,3489.25,1224.50,0.58,28.76,5.65,0.908,4.50,3.9,4.95,0.607,0.447,0.224,20
Vision,211.60,2166.50,528.10,0.29,12.32,0.00,0.958,3.80,4.6,4.90,0.696,0.503,0.308,20


Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_XAI_Stahl_SoSe26/results/eval_summary.csv


In [12]:
print('=== EMPFEHLUNG ===' )
print()

if len(judge_df) > 0:
    judge_total = judge_df.groupby('pipeline_label')[['faithfulness','clarity','completeness']].mean().sum(axis=1)
    best_pipeline = judge_total.idxmax()
    print(f'Beste Pipeline nach LLM-Judge-Gesamtscore: {best_pipeline}')
    print()
    print('Score-Rangfolge:')
    for label, score in judge_total.sort_values(ascending=False).items():
        print(f'  {label:12s}: {score:.2f} / 15')
    print()

print('Trade-off-Analyse:')
print('  JSON→Text : niedrigster Token-Verbrauch, gute Faithfulness (Zahlen direkt verfügbar)')
print('  Vision    : mittlerer Verbrauch, LLM liest Beiträge aus Plot (kann unscharf sein)')
print('  Tool-Use  : höchster Verbrauch, LLM steuert selbst die Abfrage → potenziell beste')
print('              Vollständigkeit, da es gezielt nachhaken kann')

=== EMPFEHLUNG ===

Beste Pipeline nach LLM-Judge-Gesamtscore: JSON→Text

Score-Rangfolge:
  JSON→Text   : 14.25 / 15
  Tool-Use    : 13.35 / 15
  Vision      : 13.30 / 15

Trade-off-Analyse:
  JSON→Text : niedrigster Token-Verbrauch, gute Faithfulness (Zahlen direkt verfügbar)
  Vision    : mittlerer Verbrauch, LLM liest Beiträge aus Plot (kann unscharf sein)
  Tool-Use  : höchster Verbrauch, LLM steuert selbst die Abfrage → potenziell beste
              Vollständigkeit, da es gezielt nachhaken kann


## 8 · LLM-Judge v2 – Kalibrierter Prompt

**Verbesserungen gegenüber v1:**
- Explizite Scoring-Rubrik (1–5) mit Deduktionsregeln pro Kriterium
- `feature_values_readable` im Prompt: Judge kann genannte Zahlen gegen GT verifizieren
- Grundregel: Score 4 = Standard für korrekte/vollständige Erklärung; Score 5 nur ohne Abzüge

**Methodische Note:**
v2 ist für die Interpretation als primäre Quelle zu bevorzugen. v1 dient als Vergleich
und zeigt, dass unstrukturierte Prompts systematisch zu milden Urteilen führen.


In [13]:
import json
from pathlib import Path

# Immer initialisieren – verhindert NameError in Folge-Zellen
judge_v2_df = None

# v2-Ergebnisse laden
v2_path = RESULTS_DIR / 'eval_llm_judge_v2.json'
if not v2_path.exists():
    print('⚠️  eval_llm_judge_v2.json nicht gefunden – Zelle 11 (Judge-Calls) ausführen.')
else:
    judge_v2_df = pd.DataFrame(json.loads(v2_path.read_text()))
    for col in ['faithfulness', 'clarity', 'completeness']:
        judge_v2_df[col] = pd.to_numeric(judge_v2_df[col], errors='coerce')

    v2_summary = judge_v2_df.groupby('pipeline_label')[['faithfulness','clarity','completeness']].mean().round(2)
    v2_n = judge_v2_df.groupby('pipeline_label')[['faithfulness']].count().rename(columns={'faithfulness':'n'})
    v2_std = judge_v2_df.groupby('pipeline_label')[['faithfulness','clarity','completeness']].std().round(3)
    v2_std.columns = ['faith_std','clarity_std','complete_std']
    v2_full = v2_summary.join(v2_std).join(v2_n)
    print('v2 Judge-Scores (kalibrierter Prompt):')
    display(v2_full)

    # Ceiling-Effekt v2
    tv = judge_v2_df[['faithfulness','clarity','completeness']].count().sum()
    t5 = (judge_v2_df[['faithfulness','clarity','completeness']] == 5).sum().sum()
    t4 = (judge_v2_df[['faithfulness','clarity','completeness']] == 4).sum().sum()
    print(f'\nCeiling v2: {t5}/{tv} Scores=5 ({100*t5/tv:.0f}%),  '
          f'{t4}/{tv} Scores=4 ({100*t4/tv:.0f}%)')
    print(f'(v1 zum Vergleich: 79/87 Scores=5, 91 %)')

v2 Judge-Scores (kalibrierter Prompt):


,faithfulness,clarity,completeness,faith_std,clarity_std,complete_std,n
pipeline_label,,,,,,,
JSON→Text,3.8,4.7,5.0,0.919,0.483,0.000,10
Tool-Use,4.6,4.9,5.0,0.516,0.316,0.000,10
Vision,4.4,4.7,4.9,0.699,0.483,0.316,10



Ceiling v2: 66/90 Scores=5 (73%),  18/90 Scores=4 (20%)
(v1 zum Vergleich: 79/87 Scores=5, 91 %)


In [14]:
# Vergleich v1 vs v2: Faithfulness-Differenz (der interessanteste Befund)
if judge_v2_df is None:
    print('⚠️  judge_v2_df nicht verfügbar – zuerst Zelle 8 (v2-Laden) ausführen.')
else:
    v1_faith = judge_df.groupby('pipeline_label')['faithfulness'].mean().round(2)
    v2_faith = judge_v2_df.groupby('pipeline_label')['faithfulness'].mean().round(2)

    compare = pd.DataFrame({'v1_Faith': v1_faith, 'v2_Faith': v2_faith})
    compare['Δ (v2−v1)'] = (compare['v2_Faith'] - compare['v1_Faith']).round(2)
    compare['Interpretation'] = [
        'JSON→Text: v2 strenger (Ranking-Abweichungen erkannt)',
        'Tool-Use:  v2 fairer (hr-Werte nicht mehr als Halluzination gewertet)',
        'Vision:    v2 leicht strenger',
    ]

    # Tabelle
    print('Faithfulness v1 vs v2:')
    display(compare)

    # Balken-Diagramm: v1 vs v2 Faithfulness
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    labels = ['JSON→Text', 'Tool-Use', 'Vision']
    colors = ['#4c72b0', '#55a868', '#dd8452']
    x = range(len(labels))

    # Links: Faithfulness-Vergleich
    v1_vals = [judge_df[judge_df.pipeline_label==l]['faithfulness'].mean() for l in labels]
    v2_vals = [judge_v2_df[judge_v2_df.pipeline_label==l]['faithfulness'].mean() for l in labels]
    w = 0.35
    bars1 = axes[0].bar([i - w/2 for i in x], v1_vals, w, label='v1 (unkalibriert)', color=colors, alpha=0.5)
    bars2 = axes[0].bar([i + w/2 for i in x], v2_vals, w, label='v2 (kalibriert)', color=colors)
    axes[0].set_xticks(list(x)); axes[0].set_xticklabels(labels, rotation=10)
    axes[0].set_ylim(0, 5.5); axes[0].set_ylabel('Ø Faithfulness-Score (1–5)')
    axes[0].set_title('Faithfulness: v1 vs v2')
    axes[0].legend()
    for bars in [bars1, bars2]:
        for bar in bars:
            axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                         f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

    # Rechts: Score-Verteilung v2
    from collections import Counter
    all_scores_v2 = []
    for col in ['faithfulness','clarity','completeness']:
        all_scores_v2.extend(judge_v2_df[col].dropna().astype(int).tolist())
    cnt = Counter(all_scores_v2)
    score_vals = [cnt.get(s, 0) for s in range(1, 6)]
    axes[1].bar(range(1, 6), score_vals, color='#4c72b0')
    for i, v in enumerate(score_vals):
        axes[1].text(i+1, v+0.5, str(v), ha='center', fontsize=9)
    axes[1].set_xlabel('Score'); axes[1].set_ylabel('Anzahl')
    axes[1].set_title('Score-Verteilung v2 (alle 90 Einzelscores)')
    axes[1].set_xticks(range(1,6))

    plt.tight_layout()
    out = RESULTS_DIR / 'eval_judge_v1_v2_comparison.png'
    plt.savefig(out, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Gespeichert: {out}')

Faithfulness v1 vs v2:


,v1_Faith,v2_Faith,Δ (v2−v1),Interpretation
pipeline_label,,,,
JSON→Text,4.35,3.8,-0.55,JSON→Text: v2 strenger (Ranking-Abweichungen e...
Tool-Use,4.50,4.6,0.10,Tool-Use: v2 fairer (hr-Werte nicht mehr als ...
Vision,3.80,4.4,0.60,Vision: v2 leicht strenger


Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_XAI_Stahl_SoSe26/results/eval_judge_v1_v2_comparison.png


/var/folders/z8/rlz8vq9j2cs6wj25qj_g6y700000gn/T/ipykernel_69734/3245954847.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9 · Judge v3 – Opus 4.8 (unabhängiges Modell)

**Motivation:** v1 und v2 verwendeten `claude-sonnet-4-6` sowohl zur Erklärungsgenerierung
als auch als Judge – bekannter Self-Preference-Bias (Panickssery et al., 2024).
v3 verwendet `claude-opus-4-8` als unabhängigen Judge mit identischer Kalibrierungsrubrik.

**Tool-Use Ground Truth:** Für Pipeline 06 erhält der Judge den vollständigen Tool-Call-Trace
(Aufrufe + Ergebnisse) als Teil von `ground_truth`. Per Tool abgerufene Zahlen
(PD-Kurven, Percentile, Kontrafaktika) sind damit für den Judge verifizierbar.

In [15]:
OPUS_MODEL = 'claude-opus-4-8'

opus_path = RESULTS_DIR / 'eval_llm_judge_opus.json'

if not opus_path.exists():
    print(f'Starte Opus-Judge ({OPUS_MODEL}) für alle {len(df)} Einträge …')
    opus_rows = []
    total_in, total_out = 0, 0
    MAX_RETRIES = 3

    for _, row in df.iterrows():
        prompt = build_judge_prompt(row.to_dict(), row['xai_model'], row['instance_id'])

        scores, in_tok, out_tok, raw = {}, 0, 0, ''
        for attempt in range(MAX_RETRIES):
            response = ask_text(
                prompt,
                system=JUDGE_SYSTEM,
                model=OPUS_MODEL,
                max_tokens=600,
                cache_system=True,
            )
            usage  = response.get('usage', {})
            in_tok = usage.get('input_tokens', 0)
            out_tok = usage.get('output_tokens', 0)
            raw    = response['content'][0]['text'].strip()
            scores = _parse_judge_response(raw)
            if all(scores.get(k) is not None for k in ['faithfulness', 'clarity', 'completeness']):
                break
            print(f'  ⚠️  Retry {attempt+1}/{MAX_RETRIES}: {row["pipeline_label"]} {row["xai_model"]} inst={row["instance_id"]}')

        total_in += in_tok; total_out += out_tok
        opus_rows.append({
            'pipeline_label': row['pipeline_label'],
            'xai_model':      row['xai_model'],
            'instance_id':    row['instance_id'],
            'faithfulness':   scores.get('faithfulness', None),
            'clarity':        scores.get('clarity', None),
            'completeness':   scores.get('completeness', None),
            'reasoning': {
                'faithfulness':  scores.get('faithfulness_reasoning', ''),
                'clarity':       scores.get('clarity_reasoning', ''),
                'completeness':  scores.get('completeness_reasoning', ''),
            },
            'raw_response': raw,
        })
        print(f'  {row["pipeline_label"]:12s} {row["xai_model"]} inst={row["instance_id"]:4d}  '
              f'F={scores.get("faithfulness","?")} '
              f'C={scores.get("clarity","?")} '
              f'Co={scores.get("completeness","?")}  in={in_tok}')

    opus_path.write_text(json.dumps(opus_rows, indent=2, ensure_ascii=False))
    print(f'\nGesamt: input={total_in}  output={total_out}')
else:
    print(f'Opus-Ergebnisse aus Cache geladen: {opus_path.name}')
    print('(Cache löschen zum Neuberechnen)')

judge_opus_df = pd.DataFrame(json.loads(opus_path.read_text()))
for col in ['faithfulness', 'clarity', 'completeness']:
    judge_opus_df[col] = pd.to_numeric(judge_opus_df[col], errors='coerce')

opus_summary = judge_opus_df.groupby('pipeline_label')[['faithfulness','clarity','completeness']].mean().round(2)
opus_std     = judge_opus_df.groupby('pipeline_label')[['faithfulness','clarity','completeness']].std().round(3)
opus_n       = judge_opus_df.groupby('pipeline_label')[['faithfulness']].count().rename(columns={'faithfulness': 'n'})
opus_std.columns = ['faith_std','clarity_std','complete_std']
print(f'\nOpus {OPUS_MODEL} Judge-Scores (kalibrierter Prompt, unabhängiges Modell, Tool-Trace inkludiert):')
display(opus_summary.join(opus_std).join(opus_n))

tv = judge_opus_df[['faithfulness','clarity','completeness']].count().sum()
t5 = (judge_opus_df[['faithfulness','clarity','completeness']] == 5).sum().sum()
t4 = (judge_opus_df[['faithfulness','clarity','completeness']] == 4).sum().sum()
t3 = (judge_opus_df[['faithfulness','clarity','completeness']] <= 3).sum().sum()
print(f'\nCeiling Opus: 5→{t5} ({100*t5/tv:.0f}%), 4→{t4} ({100*t4/tv:.0f}%), ≤3→{t3} ({100*t3/tv:.0f}%)')

Starte Opus-Judge (claude-opus-4-8) für alle 60 Einträge …
  JSON→Text    XGB inst= 224  F=3 C=5 Co=5  in=1327
  JSON→Text    XGB inst= 580  F=5 C=5 Co=5  in=1226
  JSON→Text    XGB inst=1041  F=3 C=5 Co=5  in=1266
  JSON→Text    XGB inst=1481  F=4 C=5 Co=5  in=1264
  JSON→Text    XGB inst=1677  F=5 C=5 Co=5  in=1285
  JSON→Text    XGB inst=2058  F=5 C=5 Co=5  in=1265
  JSON→Text    XGB inst=2510  F=4 C=5 Co=5  in=1295
  JSON→Text    XGB inst=3543  F=5 C=5 Co=5  in=1282
  JSON→Text    XGB inst=3847  F=3 C=4 Co=5  in=1290
  JSON→Text    XGB inst=4454  F=5 C=5 Co=5  in=1305
  JSON→Text    EBM inst= 224  F=5 C=5 Co=5  in=1343
  JSON→Text    EBM inst= 580  F=5 C=5 Co=5  in=1293
  JSON→Text    EBM inst=1041  F=5 C=5 Co=5  in=1277
  JSON→Text    EBM inst=1481  F=3 C=5 Co=5  in=1268
  JSON→Text    EBM inst=1677  F=5 C=5 Co=5  in=1377
  JSON→Text    EBM inst=2058  F=4 C=5 Co=5  in=1281
  JSON→Text    EBM inst=2510  F=5 C=5 Co=5  in=1297
  JSON→Text    EBM inst=3543  F=5 C=5 Co=5  in=1305
  JSO

,faithfulness,clarity,completeness,faith_std,clarity_std,complete_std,n
pipeline_label,,,,,,,
JSON→Text,4.4,4.95,5.0,0.821,0.224,0.0,20
Tool-Use,4.7,4.90,5.0,0.571,0.308,0.0,20
Vision,4.2,4.80,5.0,0.768,0.410,0.0,20



Ceiling Opus: 5→148 (82%), 4→23 (13%), ≤3→9 (5%)


In [ ]:
# Dreifach-Vergleich: v1 · v2 · Opus
if judge_v2_df is None or 'judge_opus_df' not in dir():
    print('⚠️  judge_v2_df oder judge_opus_df nicht verfügbar – zuerst Zellen 8 und 9 ausführen.')
else:
    labels_ord = ['JSON→Text', 'Tool-Use', 'Vision']
    judge_versions = [
        ('v1 Sonnet\n(unkalibriert)', judge_df,       'lightgrey', '#888'),
        ('v2 Sonnet\n(kalibriert)',   judge_v2_df,    '#7fbadf',   '#4c72b0'),
        (f'v3 Opus\n(kalibriert)',    judge_opus_df,  '#4c72b0',   '#1a3a6e'),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax_i, (crit, title) in enumerate([('faithfulness', 'Faithfulness'),
                                           ('clarity',      'Clarity'),
                                           ('completeness', 'Completeness')]):
        x = np.arange(len(labels_ord)); w = 0.25
        for j, (vlabel, df_, face, edge) in enumerate(judge_versions):
            vals = [df_[df_.pipeline_label == l][crit].mean() for l in labels_ord]
            bars = axes[ax_i].bar(x + (j - 1) * w, vals, w, label=vlabel,
                                   color=face, edgecolor=edge, linewidth=0.8)
            for bar in bars:
                h = bar.get_height()
                if not np.isnan(h):
                    axes[ax_i].text(bar.get_x() + bar.get_width() / 2, h + 0.06,
                                    f'{h:.2f}', ha='center', fontsize=7)
        axes[ax_i].set_xticks(x); axes[ax_i].set_xticklabels(labels_ord, fontsize=9)
        axes[ax_i].set_ylim(0, 5.9); axes[ax_i].set_ylabel('Ø Score (1–5)', fontsize=9)
        axes[ax_i].set_title(title, fontsize=11)
        axes[ax_i].axhline(4, color='#aaa', linestyle='--', linewidth=0.7, alpha=0.6)
        if ax_i == 0:
            axes[ax_i].legend(fontsize=8, loc='lower right')

    opus_label = OPUS_MODEL if 'OPUS_MODEL' in dir() else 'Opus'
    plt.suptitle(f'LLM-as-Judge: v1 Sonnet (unkalibriert)  ·  v2 Sonnet (kalibriert)  ·  v3 {opus_label} (kalibriert, Tool-Trace)',
                 y=1.02, fontsize=10)
    plt.tight_layout()
    out = RESULTS_DIR / 'eval_judge_all_versions.png'
    plt.savefig(out, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Gespeichert: {out}')

    print()
    print('Kernbefunde:')
    print('  Completeness: alle Judges einig → 5.0 (robuste Aussage)')
    print('  Faithfulness: Tool-Use führt in v2/v3 (kalibriert); Vision streut stark je nach Judge-Version')
    print(f'  Tool-Use Clarity Opus (mit Tool-Trace): '
          f'{judge_opus_df[judge_opus_df.pipeline_label=="Tool-Use"]["clarity"].mean():.2f}'
          f'  (kein struktureller Abzug mehr)')

## 7 · Methodische Einschränkungen

Die folgenden Punkte sind bei der Interpretation der Ergebnisse zu berücksichtigen:

| # | Einschränkung | Konsequenz |
|---|---|---|
| 1 | **Ceiling-Effekt**: Judge vergibt fast ausschließlich 4–5/5 | Clarity und Completeness diskriminieren nicht zwischen Pipelines; Radar-Diagramm suggeriert mehr Differenzierung als vorhanden |
| 2 | **Self-Preference-Bias**: v1/v2-Judge-Modell = Generation-Modell (`claude-sonnet-4-6`) | Mögliche Bevorzugung des eigenen Stils; literaturbekannter Bias (Self-Preference) nicht ausschließbar → v3 Opus als unabhängiges Modell |
| 3 | **n = 10–20 pro Pipeline** | Geringe statistische Power; Mittelwertunterschiede zwischen Pipelines nicht inferenzstatistisch absicherbar |
| 4 | **Kein Repeated Sampling** | LLM-Variabilität nicht gemessen; Consistency-Metrik fehlt trotz ursprünglicher Planung |
| 5 | **Convenience-Sampling (v2)**: Instanzen [42, 100, 250, 500, 1337] | Nicht systematisch nach cnt-Quantilen gewählt; v1 und v3 verwenden stratifiziertes Design (10 Quintil-Instanzen) |
| 6 | **Zufälliger Train/Test-Split** bei Zeitreihendaten | Mögliche temporale Leakage; R² > 0,95 teilweise dadurch erklärbar; für XAI-Demonstration vertretbar, aber Generalisierungsaussagen sind eingeschränkt |
| 7 | **Selection-Bias Ichmoukhamedov-Metriken** (→ Notebook 08) | RA/SA/VA berechnen sich nur über Features, die das LLM selbst erwähnt hat; unerwähnte Top-K-Features werden nicht bestraft → metrisches Ergebnis überschätzt echte Faithfulness |

**Fazit für die Belegarbeit:**  
Die Ergebnisse sind als explorative Demonstration zu verstehen. Statistisch robuste Aussagen über Pipeline-Unterschiede erfordern größere Stichproben (n ≥ 30 pro Pipeline), einen unabhängigen Judge und Repeated Sampling.

In [17]:
# Ceiling-Effekt quantifizieren: Wie viele Scores sind = 5?
if len(judge_df) > 0:
    total_valid = judge_df[['faithfulness', 'clarity', 'completeness']].count().sum()
    total_5     = (judge_df[['faithfulness', 'clarity', 'completeness']] == 5).sum().sum()
    total_4     = (judge_df[['faithfulness', 'clarity', 'completeness']] == 4).sum().sum()
    print(f"Ceiling-Effekt: {total_5}/{total_valid} Scores = 5 ({100*total_5/total_valid:.0f}%)")
    print(f"               {total_4}/{total_valid} Scores = 4 ({100*total_4/total_valid:.0f}%)")
    print(f"               {total_valid - total_4 - total_5} Scores ≤ 3")
    print()
    print("Std-Abweichungen der Judge-Scores pro Pipeline:")
    display(
        judge_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']]
        .std().round(3).rename(columns=lambda c: c + '_std')
    )


Ceiling-Effekt: 112/180 Scores = 5 (62%)
               54/180 Scores = 4 (30%)
               14 Scores ≤ 3

Std-Abweichungen der Judge-Scores pro Pipeline:


,faithfulness_std,clarity_std,completeness_std
pipeline_label,,,
JSON→Text,0.745,0.308,0.000
Tool-Use,0.607,0.447,0.224
Vision,0.696,0.503,0.308
